In [1]:
from datetime import UTC, datetime

import polars as pl

import nwec.utility_reporting.arrearages
import nwec.utility_reporting.num_arrearages
import nwec.utils.excel
from nwec.constants import RAW_UTILITY_DATA, Utility

YEAR = 2024
QUARTER = 4
NUM_MONTHS = 12
COLS_PER_MONTH = 1
SHEET_SEARCH_STRING = "past due balances"
spreadsheet = RAW_UTILITY_DATA / str(YEAR) / f"{Utility.CNG.code}_{YEAR}_Q{QUARTER}.xlsx"

# Number of Arrearages

In [2]:
sheet_index = nwec.utils.excel.get_sheet_index_from_name(spreadsheet, SHEET_SEARCH_STRING)
df = pl.read_excel(spreadsheet, sheet_id=sheet_index, has_header=False)
_, start_index = nwec.utils.excel.find_unpromoted_header(df, "number of customers")
num_arrearages_df = df.select(df.columns[start_index : start_index + NUM_MONTHS * COLS_PER_MONTH])
num_arrearages_df = nwec.utility_reporting.arrearages.normalize_zip_class_cols(df, num_arrearages_df)

In [3]:
date_to_zip_offset = 0
source_date_format = "%Y-%m-%d %H:%M:%S"

try:
    zip_index = nwec.utils.excel.find_unpromoted_header(num_arrearages_df, "Zip Code")
except ValueError:
    zip_index = nwec.utils.excel.find_unpromoted_header(num_arrearages_df, "Zip")
date_row = zip_index[0] - date_to_zip_offset

In [4]:
new_columns = num_arrearages_df.select(num_arrearages_df.columns[2:]).slice(date_row, 1).to_dicts()[0]
months = list({k: v for k, v in new_columns.items() if v is not None}.values())
months = months[:NUM_MONTHS]
new_columns = new_columns | {"Zip Code": "Zip Code", "Customer Class": "Customer Class"}

In [5]:
num_arrearages_df = num_arrearages_df.rename(new_columns)
num_arrearages_df = num_arrearages_df.select(num_arrearages_df.columns[: len(new_columns)])
num_arrearages_df = num_arrearages_df.filter(~pl.all_horizontal(pl.all().is_null()))
num_arrearages_df = num_arrearages_df.with_columns(
    [pl.col(col).cast(pl.Float64, strict=False) for col in num_arrearages_df.columns[2:]]
)
num_arrearages_df = num_arrearages_df.with_columns(pl.col("Zip Code").str.strip_chars(" ").cast(pl.Int32, strict=False))

# Filter out non-residential classes
num_arrearages_df = num_arrearages_df.filter(pl.col("Customer Class").str.contains(r"(?i)res"))

# Drop extraneous zip code and customer class columns
num_arrearages_df = pl.concat(
    [num_arrearages_df.select("Zip Code"), num_arrearages_df.drop(pl.selectors.matches(r"(?i)zip|customer class"))],
    how="horizontal",
)

In [6]:
new_column_names = [num_arrearages_df.columns[0]]  # Keep the first column name as is
for col in num_arrearages_df.columns[1:]:
    date_obj = datetime.strptime(col, source_date_format).replace(tzinfo=UTC)
    new_column_names.append(f"{date_obj.strftime("%m %Y")}")
num_arrearages_df = num_arrearages_df.rename(dict(zip(num_arrearages_df.columns, new_column_names, strict=False)))

In [7]:
num_arrearages_df = num_arrearages_df.unpivot(index="Zip Code")
num_arrearages_df = num_arrearages_df.with_columns(pl.col("variable").str.split_exact(" ", 1)).unnest("variable")
num_arrearages_df = num_arrearages_df.rename({"field_0": "Month", "field_1": "Year", "value": "Number of Arrearages"})
num_arrearages_df = num_arrearages_df.cast({"Year": pl.Int32, "Month": pl.Int32})
num_arrearages_df = num_arrearages_df.with_columns(pl.col("Number of Arrearages").fill_null(0))

In [8]:
nwec.utility_reporting.num_arrearages.save_processed_arrearages(num_arrearages_df, Utility.CNG, YEAR, QUARTER)